# 0-导入所需库

In [141]:
import os                                                                                   #os处理文件路径
import glob                                                                                 #glob匹配目录下以POSCAR开头的文件
import numpy as np                                                                          #numpy数学计算
from pymatgen.core import Structure                                                         #pymatgen处理晶体结构
from collections import Counter, defaultdict                                                #counter, defaultdict统计元素出现次数
from scipy.spatial import cKDTree                                                           #cKDTree查找近邻原子
import pandas as pd                                                                         #pandas处理数据为表格形式
import itertools                                                                            #itertools为高效循环创建迭代器
import pickle                                                                               #pickle是python专用的持续化模块，将对象转化为字节流

# 1-构型熵计算
整体组成熵：基于阳离子元素的种类和数量

局域环境构型熵：基于各阳离子位点的近邻环境分析

## 1.1 使用os.path.join管理输入结构POSCAR的文件路径；pymatgen创建structure对象
POSCAR结构目录（file_path）：D:\github\Disorder_learner\structure_file

阳离子选择参考文献：(Adv. Mater. 2025, 37, e08717)   (Energy Environ. Sci., 2020, 13, 4450-4497)，排除高价、有毒、半径过大，以及在锂电中不常见的金属元素

In [142]:
work_path = os.getcwd()
structure_dir = os.path.join(work_path, "structure_file")                                    #结构文件目录

all_cation_elements = [
    'Li', 'Mg',                                                                              # alkail metal                      
    'Ti', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn',                                                # 3d transition metal 
    'Zr', 'Nb', 'Mo', 'Ru',                                                                  # 4d/5d transition metal   
    'Al', 'Sn'                                                                               # post transition metal 
]

def find_poscar_files(structure_dir):
    poscar_patterns = [os.path.join(structure_dir, "POSCAR*")]                               # 查找POSCAR开头的文件作为输入结构    
    poscar_files = []
    for pattern in poscar_patterns:
        poscar_files.extend(glob.glob(pattern))
    return sorted(list(set(poscar_files)))

def read_structure(file_path):    
    struct = Structure.from_file(file_path)                                                  # 读取结构文件    
    file_name = os.path.basename(file_path)
    struct_name = file_name.replace("POSCAR", "").replace("_", " ").strip() 
    
    lattice = struct.lattice                                                                 # 获取晶胞参数
    a, b, c = lattice.a, lattice.b, lattice.c

    return struct, struct_name, lattice, (a, b, c)

## 1.2 按照整个结构中各阳离子的数量，计算整体组成熵

 组成熵公式：$ S_{config} = -R \left[\left( \sum_{{i}=1}^{{n}} x_{{i}} \ln x_{{i}} \right)_{\text{cation-site}} \right] $           其中，R为摩尔气体常数[8.314 J (mol−1 K−1)]，n表示组分数，xi代表阳离子的摩尔分数。
 
 (Acta Mater., 122, 2017, pp. 448-511)
 
 (Small Methods, 2023, 7, e2300152)  (enthalpy and synergy. Scr. Mater. 2020, 187, 43−48)  (ACS Materials Lett. 2021, 3, 160−170)


In [143]:
def calculate_global_entropy(struct, all_cation_elements):
    all_cation_species = []
    for site in struct:
        element = site.species.element_composition.elements[0].symbol
        if element in all_cation_elements:
            all_cation_species.append(element)
    
    element_counts = Counter(all_cation_species)                                            #统计元素组成 组成字符串
    total_cations = len(all_cation_species)  
    element_composition = ", ".join([f"{elem}:{count}" for elem, count in sorted(element_counts.items())])    

    R = 8.314                                                                               # 理想气体常数 J/mol·K
    global_entropy = 0.0                                                                    # 初始化理想气体常数和熵值
    for element, count in element_counts.items():
        x_i = count / total_cations                                                         # 计算阳离子元素的摩尔分数
        if x_i > 0:
            global_entropy -= x_i * np.log(x_i)                                             # 只对正数进行计算，避免log(0)错误
    sconfig_global = R * global_entropy                                                     # 将标准化熵值乘以理想气体常数得到实际熵值

    return {
        'element_composition': element_composition,
        'configurational entropy': sconfig_global,
        'total_cations': total_cations,
        'element_counts': element_counts
    }

## 1.3 各阳离子位点的局域环境分析
在ab平面内创建镜像位点模拟周期性，确保边缘阳离子的局域环境检索（cKDTree）
|

截断半径（cutoff_radius）设为3.8Å，用来区分近邻位点；
c轴方向容差（z_tolerance）设为0.2，仅识别同一层ab平面内的近邻位点；
阳离子配位数（target_coordination）设为6，默认取6个近邻阳离子

In [144]:
def analyze_local_environment(struct, all_cation_elements, lattice_params, cutoff_radius=3.8, z_tolerance=0.2, target_coordination=6):
    a, b, c = lattice_params                                                                # 获取晶格参数
    
    cation_sites = []                                                                       # 收集所有阳离子位点
    for i, site in enumerate(struct):
        element = site.species.element_composition.elements[0].symbol
        if element in all_cation_elements:
            cation_sites.append({
                'index': i,                                                                 # 阳离子位点在结构中的索引
                'element': element,
                'frac_coords': site.frac_coords,                                            # 分数坐标
                'coords': site.coords,                                                      # 笛卡尔坐标
            })
    layers_x = int(np.ceil(cutoff_radius / a)) + 1                                          # 通过创建镜像位点以模拟晶体结构ab平面的周期性
    layers_y = int(np.ceil(cutoff_radius / b)) + 1
    
    all_cation_coords = []
    all_cation_elements_list = []
    all_cation_frac_z = []

    for site in cation_sites:                                                               #记录原始坐标
        base_coords = site['coords']
        base_element = site['element']
        base_frac_z = site['frac_coords'][2]

        all_cation_coords.append(base_coords)                                              # 添加中心单元位点
        all_cation_elements_list.append(base_element)
        all_cation_frac_z.append(base_frac_z)
                                                                                           # 在ab平面内创建周期性镜像位点，跳过该中心位点
        for dx in range(-layers_x, layers_x + 1):                                          # +1确保能覆盖截断半径内的所有可能近邻
            for dy in range(-layers_y, layers_y + 1):
                if dx == 0 and dy == 0:
                    continue
                
                mirror_coords = base_coords.copy()                                         # 在ab平面内沿晶格向量方向平移
                mirror_coords[0] += dx * a
                mirror_coords[1] += dy * b
                
                all_cation_coords.append(mirror_coords)
                all_cation_elements_list.append(base_element)
                all_cation_frac_z.append(base_frac_z)
                                                                                          # 创建KDTree进行近邻搜索
    all_cation_coords = np.array(all_cation_coords)
    kdtree = cKDTree(all_cation_coords)    
    all_environments = []
    valid_sites = 0    
    for site in cation_sites:                                                             # 分析各阳离子位点的局域环境
        site_coords = site['coords']
        site_frac_z = site['frac_coords'][2]
        
        k = min(len(all_cation_coords), target_coordination * 3)                          # 收集近邻阳离子位点
        distances, indices = kdtree.query(site_coords, k=k)
        
        ab_plane_neighbors = []                                                           # 筛选ab平面内的近邻位点
        for j, idx in enumerate(indices):
            if j == 0:                                                                    # 跳过该中心位点
                continue
            
            distance = distances[j]
            if distance > cutoff_radius:                                                  # 根据截断半径进行近邻筛选
                continue
            
            neighbor_frac_z = all_cation_frac_z[idx]
            z_diff = abs(neighbor_frac_z - site_frac_z)                                   # 根据c方向容差进行ab平面内筛选
            z_diff_periodic = min(z_diff, 1 - z_diff) * c          
            if z_diff_periodic > z_tolerance:
                continue
            
            neighbor_element = all_cation_elements_list[idx] 
            if neighbor_element in all_cation_elements:                                   # 检查是否为阳离子
                ab_plane_neighbors.append({
                    'element': neighbor_element,
                    'distance': distance,
                })
        
        ab_plane_neighbors.sort(key=lambda x: x['distance'])                              # 按距离排序并选择近邻位点
        selected_neighbors = ab_plane_neighbors[:target_coordination]
        
        if len(selected_neighbors) >= target_coordination:                                # 每个阳离子取6个近邻位点
            valid_sites += 1
            neighbor_elements = [n['element'] for n in selected_neighbors]
            fingerprint = tuple(sorted(neighbor_elements))
            all_environments.append(fingerprint)

    return {
        'all_environments': all_environments,
        'valid_sites': valid_sites,
        'total_sites': len(cation_sites),
        'cation_sites': cation_sites
    }

## 1.4 局域无序构型熵计算

局域构型熵公式：$S_{local} = -R \sum_{j=1}^{m} p_j \ln p_j$   

(m = 局域环境类型的总数; pj = 第 j 种局域环境出现的概率）

In [145]:
def calculate_local_disorder(all_environments, valid_sites):
    environment_counts = Counter(all_environments)                                         # 统计不同局域环境类型的出现频率
    num_local_environments = len(environment_counts)

    local_shannon_entropy = 0.0                                                            # 计算各阳离子位点的局域构型熵
    for env, count in environment_counts.items():
        p_env = count / valid_sites
        if p_env > 0:
            local_shannon_entropy -= p_env * np.log(p_env)

    R = 8.314                                                                              # 理想气体常数 J/mol·K
    local_disorder = R * local_shannon_entropy
    return {
        'local configurational entropy': local_disorder,
        'Effective site Types of Local Environment': num_local_environments
    }

## 1.5 批量POSCAR结构的构型熵计算

In [133]:
#base code block:1.1,1.2,1.3,1.4
def calculate_sconfig(file_path, all_cation_elements):                                                                            # 整合上面所有步骤
    struct, struct_name, lattice, lattice_params = read_structure(file_path)                                                      # 1.1 读取结构
    global_results = calculate_global_entropy(struct, all_cation_elements)                                                        # 1.2 计算整体组成熵
    local_env_results = analyze_local_environment(struct, all_cation_elements, lattice_params)                                    # 1.3 分析局域环境
    local_disorder_results = calculate_local_disorder(local_env_results['all_environments'],local_env_results['valid_sites'])     # 1.4 计算局域构型熵

    results = {
        'Elemental composition': global_results['element_composition'],
        'Configurational entropy': round(global_results['configurational entropy'], 4),
        'Local configurational entropy': round(local_disorder_results['local configurational entropy'], 4),
        'Cation sites': f"{local_env_results['valid_sites']}/{local_env_results['total_sites']}",
        'Effective site Types of Local Environment': local_disorder_results['Effective site Types of Local Environment']
    }   
    return results

## 1.6 输出计算结果

In [134]:
def analyze_all_structures():
    poscar_files = find_poscar_files(structure_dir)
    all_results = []
    
    for file_path in poscar_files:
        result = calculate_sconfig(file_path, all_cation_elements)
        all_results.append(result)

    df = pd.DataFrame(all_results)                                                         # 创建并显示DataFrame
    display_columns = [
        'Elemental composition',
        'Configurational entropy', 
        'Local configurational entropy',
        'Cation sites',
        'Effective site Types of Local Environment'
    ]
    
    display(df[display_columns])    

    return df

final_results = analyze_all_structures()

NameError: name 'all_cation_elements' is not defined

# 2-优化模型构建

## 2.1 生成价态平衡的化学式组合
化学通式：Li₁.₂Mn₀.₂MₓMᵧMzO₂（固定Mn含量为0.2，M₁, M₂, M₃为三种不同的过渡金属）

In [135]:
element_valences = {                                                                     #定义阳离子的价态字典
    'Mg': 2, 'Cu': 2, 'Ni': 2, 'Zn': 2,
    'Fe': 3, 'Co': 3, 'Al': 3, 
    'Zr': 4, 'Ti': 4, 'Mn': 4, 'Ru': 4, 'Sn': 4,
    'Nb': 5, 'Mo': 6
}
def generate_optimized_formulas(step=0.1):                                               #约束元素含量 x+y+z = 0.6，步长设为 0.1                                                                                      
    total_steps = int(round(0.6 / step))                                                 #INT是取整函数，round()方法返回浮点数x的四舍五入值。将 0.6 和 step 转化为整数步数，避免浮点运算带来的精度丢失
    available_elements = [e for e in element_valences.keys() if e != 'Mn']               #创建可用元素列表 available_elements，排除固定的 Mn
    proportion_combinations = []
    for x_steps in range(1, total_steps - 1):                                            #对于所有元素组合，预生成所有可能的比例
        for y_steps in range(1, total_steps - x_steps):
            z_steps = total_steps - x_steps - y_steps
            if z_steps > 0:
                proportion_combinations.append((x_steps, y_steps, z_steps))           
    valid_data = []
    
    for elements in itertools.combinations(available_elements, 3):                       #遍历所有可能的三种元素组合
        e1, e2, e3 = elements
        v1, v2, v3 = element_valences[e1], element_valences[e2], element_valences[e3]

        for (sx, sy, sz) in proportion_combinations:                                     # 测试化学式比例是否满足电荷平衡
            target_valence_steps = int(round(2.0 / step))
            
            if (v1 * sx + v2 * sy + v3 * sz) == target_valence_steps:
                x, y, z = sx * step, sy * step, sz * step
                
                valid_data.append({
                    'Element_1': e1, 'Proportion_1': x, 'Valence_1': v1,
                    'Element_2': e2, 'Proportion_2': y, 'Valence_2': v2,
                    'Element_3': e3, 'Proportion_3': z, 'Valence_3': v3,
                    'Formula': f"Li1.2Mn0.2{e1}{x:.2f}{e2}{y:.2f}{e3}{z:.2f}O2"
                })
                
    return pd.DataFrame(valid_data)

df_formulas = generate_optimized_formulas(step=0.1)                                      #执行输出代码                             
print(f"共生成 {len(df_formulas)} 个电荷平衡的化学式组合")
df_formulas

共生成 284 个电荷平衡的化学式组合


,Element_1,Proportion_1,Valence_1,Element_2,Proportion_2,Valence_2,Element_3,Proportion_3,Valence_3,Formula
0,Mg,0.1,2,Cu,0.1,2,Zr,0.4,4,Li1.2Mn0.2Mg0.10Cu0.10Zr0.40O2
1,Mg,0.1,2,Cu,0.1,2,Ti,0.4,4,Li1.2Mn0.2Mg0.10Cu0.10Ti0.40O2
2,Mg,0.1,2,Cu,0.1,2,Ru,0.4,4,Li1.2Mn0.2Mg0.10Cu0.10Ru0.40O2
3,Mg,0.1,2,Cu,0.1,2,Sn,0.4,4,Li1.2Mn0.2Mg0.10Cu0.10Sn0.40O2
4,Mg,0.1,2,Cu,0.3,2,Mo,0.2,6,Li1.2Mn0.2Mg0.10Cu0.30Mo0.20O2
...,...,...,...,...,...,...,...,...,...,...
279,Al,0.4,3,Zr,0.1,4,Ru,0.1,4,Li1.2Mn0.2Al0.40Zr0.10Ru0.10O2
280,Al,0.4,3,Zr,0.1,4,Sn,0.1,4,Li1.2Mn0.2Al0.40Zr0.10Sn0.10O2
281,Al,0.4,3,Ti,0.1,4,Ru,0.1,4,Li1.2Mn0.2Al0.40Ti0.10Ru0.10O2
282,Al,0.4,3,Ti,0.1,4,Sn,0.1,4,Li1.2Mn0.2Al0.40Ti0.10Sn0.10O2


## 2.2 替换过渡金属层内原子得到结构

In [136]:
def batch_generate_structures(df, baseline_path, output_dir="Generated_VASP"):

    os.makedirs(output_dir, exist_ok=True)
    base_struct = Structure.from_file(baseline_path)                                               # 基准结构在本目录下
    tm_indices = [i for i, site in enumerate(base_struct) if site.species_string == "Co"]          # 替换过渡金属层的Co位点
    total_tm = len(tm_indices)                                                                     # len函数返回可迭代对象中的元素数量，存储于变量total_tm

    multiplier = total_tm / 0.8                                                                    # 计算过渡金属层内每种元素比例
    for idx, row in df.iterrows():                                                                 # 计算每种元素的原子替换数
        n1 = int(round(row['Proportion_1'] * multiplier))
        n2 = int(round(row['Proportion_2'] * multiplier))
        n3 = int(round(row['Proportion_3'] * multiplier))        
        n_mn = int(round(0.2 * multiplier))                                                        # 固定Mn含量
        
        pool = (['Mn']*n_mn + [row['Element_1']]*n1 + 
                [row['Element_2']]*n2 + [row['Element_3']]*n3)
        
        random.seed(idx)                                                                           # 用固定的 idx 初始化随机数生成器，具有可重复性
        random.shuffle(pool)                                                                       # 打乱
        
        new_struct = base_struct.copy()                                                            # 创建副本不影响基准结构
        for site_idx, new_elem in zip(tm_indices, pool):                                           # 将tm_indices和 pool这两个对象配对替换原子
            new_struct.replace(site_idx, new_elem)
            
        filename = f"POSCAR_{idx:03d}_{row['Formula']}".replace(".", "_")                          # 生成结构命名格式
        new_struct.to(fmt="poscar", filename=os.path.join(output_dir, filename))                   # 生成结构存到文件夹
    print(f"共生成 {len(df)} 个 VASP 结构至 {output_dir}")

df_formulas = generate_optimized_formulas(step=0.1)                                                # 运行第一步：生成化学式
display(df_formulas.head())
baseline_file = os.path.join(work_path, "POSCAR.cif")                                              # 运行第二步：在基准结构的基础上，生成大量替换后结构
batch_generate_structures(df_formulas, baseline_file)

,Element_1,Proportion_1,Valence_1,Element_2,Proportion_2,Valence_2,Element_3,Proportion_3,Valence_3,Formula
0,Mg,0.1,2,Cu,0.1,2,Zr,0.4,4,Li1.2Mn0.2Mg0.10Cu0.10Zr0.40O2
1,Mg,0.1,2,Cu,0.1,2,Ti,0.4,4,Li1.2Mn0.2Mg0.10Cu0.10Ti0.40O2
2,Mg,0.1,2,Cu,0.1,2,Ru,0.4,4,Li1.2Mn0.2Mg0.10Cu0.10Ru0.40O2
3,Mg,0.1,2,Cu,0.1,2,Sn,0.4,4,Li1.2Mn0.2Mg0.10Cu0.10Sn0.40O2
4,Mg,0.1,2,Cu,0.3,2,Mo,0.2,6,Li1.2Mn0.2Mg0.10Cu0.30Mo0.20O2


共生成 284 个 VASP 结构至 Generated_VASP
